In [1]:
# Cell 1: Dependencies & Hardware Check
!pip install -q transformer-lens torch numpy scipy pandas scikit-learn tabulate

import torch
import warnings
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Hardware Status: {device.upper()}")
if device == "cuda":
    print(f"🎮 GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"🧠 VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 4.4 MB/s eta 0:00:00
✅ Hardware Status: CUDA
🎮 GPU Model: Tesla T4
🧠 VRAM Available: 15.64 GB


In [ ]:
import torch
import numpy as np
import json
import warnings
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression
from transformer_lens import HookedTransformer

warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B"


# ==============================================================================
# 0. SERIALIZATION & HELPERS
# ==============================================================================

def to_jsonable(obj: Any) -> Any:
    if isinstance(obj, dict): return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, np.generic): return obj.item()
    if isinstance(obj, torch.Tensor): return obj.item() if obj.ndim == 0 else obj.detach().cpu().tolist()
    if isinstance(obj, (str, int, float, bool)) or obj is None: return obj
    if hasattr(obj, "__dataclass_fields__"): return to_jsonable(asdict(obj))
    return str(obj)


def extract_context_family(ctx_str: str) -> str:
    for prefix in ["train_", "calib_", "test_", "ood_"]:
        if ctx_str.startswith(prefix): return ctx_str[len(prefix):]
    return ctx_str


# ==============================================================================
# 1. ACTION-SENSITIVE UTILITY MATRIX
# ==============================================================================

class DecoupledUtility:
    THRESH_GAIN = 0.005
    THRESH_HAZARD = -0.05
    ALL_ACTIONS = ["APPROVED", "VETO_ZERO_GAIN", "VETO_HAZARD", "VETO_EPISTEMIC"]

    @staticmethod
    def get_utility(action: str, delta_m: float) -> float:
        if delta_m >= DecoupledUtility.THRESH_GAIN:
            if action == "APPROVED": return float(delta_m)
            if action == "VETO_EPISTEMIC": return -0.005
            if action == "VETO_ZERO_GAIN": return -0.015
            if action == "VETO_HAZARD": return -0.03
        elif delta_m < DecoupledUtility.THRESH_HAZARD:
            if action == "VETO_HAZARD": return 0.05
            if action == "VETO_EPISTEMIC": return 0.03
            if action == "VETO_ZERO_GAIN": return 0.01
            if action == "APPROVED": return float(10.0 * delta_m)
        else:
            if action == "VETO_ZERO_GAIN": return 0.005
            if action == "VETO_EPISTEMIC": return 0.002
            if action == "VETO_HAZARD": return 0.0
            if action == "APPROVED": return float(delta_m)
        return -1.0

    @staticmethod
    def get_omniscient_oracle(delta_m: float) -> Tuple[str, float]:
        utils = [DecoupledUtility.get_utility(a, delta_m) for a in DecoupledUtility.ALL_ACTIONS]
        best_idx = int(np.argmax(utils))
        return DecoupledUtility.ALL_ACTIONS[best_idx], utils[best_idx]


# ==============================================================================
# 2. BAYESIAN SCM: STRICT PARAMETRIC EPISTEMIC + RESIDUAL ALEATORIC
# ==============================================================================

@dataclass
class TargetSpec:
    layer: int
    component: str
    index: int = 0
    keep_ratio: float = 0.0

    @property
    def dose(self) -> float:
        return float(np.clip(1.0 - self.keep_ratio, 0.0, 1.0))

    @property
    def key(self) -> str:
        return f"L{self.layer}_{self.component}_{self.index}"


@dataclass
class PromptItem:
    prompt: str
    target_str: str
    distractor_str: str


class PureEpistemicSelfGraph:
    def __init__(self, prior_variance: float = 0.5, base_aleatoric: float = 0.005):
        self.prior_variance = prior_variance
        self.base_aleatoric = base_aleatoric
        self.component_models: Dict[str, Dict[str, Dict[str, Any]]] = {}

    def init_node(self, target_key: str, ctx_str: str):
        family = extract_context_family(ctx_str)
        if target_key not in self.component_models:
            self.component_models[target_key] = {}
        if family not in self.component_models[target_key]:
            self.component_models[target_key][family] = {
                "mu": np.zeros(2),
                "precision": np.eye(2) / self.prior_variance,
                "cov": np.eye(2) * self.prior_variance,
                "aleatoric_var": self.base_aleatoric,
                "history": [],
                "n": 0
            }

    def update(self, target_key: str, ctx_str: str, dose: float, delta_obs: float):
        family = extract_context_family(ctx_str)
        self.init_node(target_key, ctx_str)
        node = self.component_models[target_key][family]

        x = np.array([dose, dose**2])
        node["history"].append((dose, delta_obs))

        if len(node["history"]) >= 3:
            y_vals = np.array([h[1] for h in node["history"]])
            diffs = np.diff(y_vals)
            node["aleatoric_var"] = max(self.base_aleatoric, float(0.5 * np.mean(diffs**2)))

        inv_noise = 1.0 / max(node["aleatoric_var"], 1e-6)
        prec_old = node["precision"]
        mu_old = node["mu"]

        prec_new = prec_old + inv_noise * np.outer(x, x)
        cov_new = np.linalg.inv(prec_new)
        mu_new = cov_new @ (prec_old @ mu_old + inv_noise * x * delta_obs)

        node["precision"] = prec_new
        node["cov"] = cov_new
        node["mu"] = mu_new
        node["n"] += 1

    def predict(self, targets: List[TargetSpec], ctx_str: str) -> Tuple[float, float, float, float]:
        family = extract_context_family(ctx_str)
        pred_delta = 0.0
        var_epi = 0.0
        var_ale = 0.0

        for t in targets:
            if t.dose == 0.0: continue
            self.init_node(t.key, ctx_str)
            node = self.component_models[t.key][family]
            x_target = np.array([t.dose, t.dose**2])
            pred_delta += float(x_target @ node["mu"])
            var_epi += float(x_target @ node["cov"] @ x_target)
            var_ale += float(node["aleatoric_var"])

        total_var = var_epi + var_ale
        return pred_delta, float(np.sqrt(total_var)), float(np.sqrt(var_epi)), float(np.sqrt(var_ale))


# ==============================================================================
# 3. EMPIRICAL CALIBRATOR
# ==============================================================================

class ScaledEmpiricalCalibrator:
    def __init__(self):
        self.model = LogisticRegression(multi_class="multinomial", solver="lbfgs", max_iter=300, C=1.0)
        self.is_fitted = False
        self.features: List[np.ndarray] = []
        self.labels: List[int] = []

    @staticmethod
    def extract_vector(mu: float, sig_pred: float, sig_epi: float) -> np.ndarray:
        sig_pred = max(sig_pred, 1e-6)
        z_gain = (mu - DecoupledUtility.THRESH_GAIN) / sig_pred
        z_haz = (DecoupledUtility.THRESH_HAZARD - mu) / sig_pred
        return np.array([mu, sig_pred, sig_epi, z_gain, z_haz])

    def store_sample(self, mu: float, sig_pred: float, sig_epi: float, true_delta: float):
        vec = self.extract_vector(mu, sig_pred, sig_epi)
        if true_delta >= DecoupledUtility.THRESH_GAIN: y = 0
        elif true_delta < DecoupledUtility.THRESH_HAZARD: y = 2
        else: y = 1
        self.features.append(vec)
        self.labels.append(y)

    def train_calibrator(self) -> bool:
        if len(self.labels) < 12 or len(set(self.labels)) < 3:
            self.is_fitted = False
            return False
        X = np.array(self.features)
        y = np.array(self.labels)
        self.model.fit(X, y)
        self.is_fitted = True
        return True

    def predict_beliefs(self, mu: float, sig_pred: float, sig_epi: float) -> Dict[str, float]:
        sig_pred = max(sig_pred, 1e-6)
        if not self.is_fitted:
            p_haz = float(norm.cdf((DecoupledUtility.THRESH_HAZARD - mu) / sig_pred))
            p_gain = float(1.0 - norm.cdf((DecoupledUtility.THRESH_GAIN - mu) / sig_pred))
            p_zero = max(0.0, 1.0 - p_haz - p_gain)
            return {"P_gain": p_gain, "P_zero": p_zero, "P_hazard": p_haz}

        vec = self.extract_vector(mu, sig_pred, sig_epi).reshape(1, -1)
        probs = self.model.predict_proba(vec)[0]
        classes = self.model.classes_

        p_gain = float(probs[np.where(classes == 0)[0][0]]) if 0 in classes else 0.0
        p_zero = float(probs[np.where(classes == 1)[0][0]]) if 1 in classes else 0.0
        p_hazard = float(probs[np.where(classes == 2)[0][0]]) if 2 in classes else 0.0

        total = p_gain + p_zero + p_hazard
        if total <= 0: total = 1.0
        return {"P_gain": p_gain / total, "P_zero": p_zero / total, "P_hazard": p_hazard / total}


# ==============================================================================
# 4. MECHANISTIC SENSORS INTERFACE
# ==============================================================================

class MechanisticSensorInterface:
    def __init__(self, model_name: str = MODEL_NAME, device: str = "cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        print(f"🔄 Initializing Sensors ({model_name}) on {self.device.upper()}...")
        self.model = HookedTransformer.from_pretrained(
            model_name, device=self.device, torch_dtype=torch.float32, default_padding_side="left"
        )
        self.model.eval()
        for p in self.model.parameters(): p.requires_grad_(False)

    def _build_hooks(self, targets: List[TargetSpec]):
        hooks = []
        for t in targets:
            keep = t.keep_ratio
            if t.component == "head":
                h_idx = t.index
                def head_hook_fn(act, hook, h=h_idx, r=keep):
                    act[:, -1, h, :] = act[:, -1, h, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.attn.hook_z", head_hook_fn))
            elif t.component == "mlp":
                def mlp_hook_fn(act, hook, r=keep):
                    act[:, -1, :] = act[:, -1, :] * r
                    return act
                hooks.append((f"blocks.{t.layer}.hook_mlp_out", mlp_hook_fn))
        return hooks

    def _resolve_token_id(self, token_str: str) -> int:
        toks = self.model.to_tokens(token_str, prepend_bos=False)[0]
        return int(toks[-1].item())

    @torch.no_grad()
    def measure_grounded_outcomes(
        self, items: List[PromptItem], targets: Optional[List[TargetSpec]] = None, noise_sigma: float = 0.0
    ) -> float:
        prompts = [it.prompt for it in items]
        tokens = self.model.to_tokens(prompts)
        batch_size = len(items)
        b_idx = torch.arange(batch_size, device=self.device)

        target_ids = torch.tensor([self._resolve_token_id(it.target_str) for it in items], device=self.device)
        distractor_ids = torch.tensor([self._resolve_token_id(it.distractor_str) for it in items], device=self.device)

        clean_logits = self.model(tokens)[:, -1, :].float()
        clean_m = clean_logits[b_idx, target_ids] - clean_logits[b_idx, distractor_ids]

        if not targets: return 0.0

        with self.model.hooks(fwd_hooks=self._build_hooks(targets)):
            int_logits = self.model(tokens)[:, -1, :].float()

        int_m = int_logits[b_idx, target_ids] - int_logits[b_idx, distractor_ids]
        raw_delta = float(torch.mean(int_m - clean_m).item())

        if noise_sigma > 0.0:
            return raw_delta + float(np.random.normal(0.0, noise_sigma))
        return raw_delta


# ==============================================================================
# 5. CAUSALLY ISOLATED PROBING CONTROLLER
# ==============================================================================

class IsolatedPolicyEngine:
    def __init__(self, scm: PureEpistemicSelfGraph, calibrator: ScaledEmpiricalCalibrator, policy_mode: str):
        self.scm = scm
        self.calibrator = calibrator
        self.mode = policy_mode
        self.cost_factor = 0.001
        self.lambda_probe_risk = 0.5
        self.candidates = np.array([0.10, 0.20, 0.35, 0.50, 0.65, 0.75])
        self.eval_shifts = np.array([-2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0])
        self.eval_weights = norm.pdf(self.eval_shifts)
        self.eval_weights /= np.sum(self.eval_weights)

    def _evaluate_max_expected_utility(self, mu_star: float, sigma_star: float) -> float:
        sigma_star = max(sigma_star, 1e-6)
        samples = mu_star + sigma_star * self.eval_shifts
        action_expected_utils = []
        for a in DecoupledUtility.ALL_ACTIONS:
            eu_a = sum(w * DecoupledUtility.get_utility(a, s) for w, s in zip(self.eval_weights, samples))
            action_expected_utils.append(eu_a)
        return float(max(action_expected_utils))

    def evaluate_step(
        self, target: TargetSpec, ctx_str: str, executed_doses: List[float]
    ) -> Tuple[float, bool, str, str, Dict[str, float]]:
        self.scm.init_node(target.key, ctx_str)
        node = self.scm.component_models[target.key][extract_context_family(ctx_str)]

        pred_mu, sig_pred, sig_epi, _ = self.scm.predict([target], ctx_str)
        beliefs = self.calibrator.predict_beliefs(pred_mu, sig_pred, sig_epi)

        epistemic_state = "EPISTEMICALLY_UNCERTAIN"
        if beliefs["P_hazard"] >= 0.55: epistemic_state = "CONFIDENT_HAZARD"
        elif beliefs["P_gain"] >= 0.55: epistemic_state = "CONFIDENT_GAIN"
        elif beliefs["P_zero"] >= 0.55: epistemic_state = "CONFIDENT_ZERO_GAIN"

        if len(executed_doses) >= 1 and epistemic_state != "EPISTEMICALLY_UNCERTAIN":
            return 0.0, True, epistemic_state, f"STOP_{epistemic_state}", beliefs

        current_v = self._evaluate_max_expected_utility(pred_mu, sig_pred)
        best_net_score = -float("inf")
        best_dose = self.candidates[0]

        pts, wts = np.polynomial.hermite.hermgauss(5)

        for d in self.candidates:
            if any(abs(d - ed) < 0.05 for ed in executed_doses): continue

            if len(executed_doses) >= 1 and pred_mu < -0.05 and d > 0.40:
                continue

            x_d = np.array([d, d**2])
            pred_y = float(x_d @ node["mu"])
            std_y = np.sqrt(float(x_d @ node["cov"] @ x_d) + node["aleatoric_var"])

            prec_sim = node["precision"] + (1.0 / node["aleatoric_var"]) * np.outer(x_d, x_d)
            cov_sim = np.linalg.inv(prec_sim)
            x_target = np.array([1.0, 1.0])
            sig_pred_sim = float(np.sqrt(x_target @ cov_sim @ x_target + node["aleatoric_var"]))

            expected_posterior_v = 0.0
            probe_harm_expected = 0.0
            for pt, w in zip(pts, wts):
                y_sample = pred_y + np.sqrt(2.0) * std_y * pt
                mu_sim = cov_sim @ (node["precision"] @ node["mu"] + (1.0 / node["aleatoric_var"]) * x_d * y_sample)
                mu_full_sim = float(x_target @ mu_sim)
                expected_posterior_v += (w / np.sqrt(np.pi)) * self._evaluate_max_expected_utility(mu_full_sim, sig_pred_sim)
                probe_harm_expected += (w / np.sqrt(np.pi)) * max(0.0, -y_sample)

            if self.mode == "PURE_INFO":
                info_gain = max(0.0, sig_pred - sig_pred_sim)
                net_score = info_gain - (self.cost_factor * float(d**2)) - (self.lambda_probe_risk * probe_harm_expected)
            else:
                evoi_decision = max(0.0, expected_posterior_v - current_v)
                net_score = evoi_decision - (self.cost_factor * float(d**2)) - (self.lambda_probe_risk * probe_harm_expected)

            if net_score > best_net_score:
                best_net_score = net_score
                best_dose = float(d)

        if len(executed_doses) >= 3 and best_net_score <= 0.0:
            return 0.0, True, epistemic_state, "STOP_UNECONOMIC", beliefs

        return best_dose, False, epistemic_state, "CONTINUE_PROBING", beliefs


# ==============================================================================
# 6. PROTOCOL BENCHMARK CONTROLLER (MILESTONE 17.18)
# ==============================================================================

def run_milestone_17_18():
    sensors = MechanisticSensorInterface()

    corpus = {
        "ood_syntactic_agreement": [
            PromptItem("The keys to the cabinet are kept in the", " drawer", " drawers"),
            PromptItem("The books on the wooden table belong to the", " library", " libraries"),
            PromptItem("The pictures of the old palace show the ancient", " king", " kings"),
            PromptItem("The report about the recent discoveries was presented by", " researcher", " researchers"),
        ],
        "ood_distractor_reasoning": [
            PromptItem("While Rome has the Colosseum, the Eiffel Tower stands in", " Paris", " Rome"),
            PromptItem("Even though Madrid is famous, the capital of Germany is", " Berlin", " Madrid"),
            PromptItem("Egypt has pyramids, but the Eiffel Tower was built in the city of", " Paris", " Cairo"),
        ],
        "train_factual_cap": [
            PromptItem("The capital of France is the city of", " Paris", " Rome"),
            PromptItem("The capital of Japan is the bustling city of", " Tokyo", " Kyoto"),
            PromptItem("The official capital city of Germany is", " Berlin", " Munich"),
            PromptItem("The historic capital of Italy is the city of", " Rome", " Milan"),
        ]
    }

    # استخر کالیبراسیون تجربی (۱۸ نمونه مستقل)
    calibration_pool_18 = [
        {"desc": "L18 MLP", "target": TargetSpec(18, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L19 MLP", "target": TargetSpec(19, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L21 MLP", "target": TargetSpec(21, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L23 Head 0", "target": TargetSpec(23, "head", 0, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L18 Head 1", "target": TargetSpec(18, "head", 1, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L19 Head 2", "target": TargetSpec(19, "head", 2, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L1 Head 0", "target": TargetSpec(1, "head", 0, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L8 Head 4", "target": TargetSpec(8, "head", 4, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L5 Head 2", "target": TargetSpec(5, "head", 2, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L5 Head 0", "target": TargetSpec(5, "head", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L8 Head 1", "target": TargetSpec(8, "head", 1, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L1 Head 2", "target": TargetSpec(1, "head", 2, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L16 MLP", "target": TargetSpec(16, "mlp", 0, keep_ratio=0.0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L22 Head 0", "target": TargetSpec(22, "head", 0, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L23 Head 1", "target": TargetSpec(23, "head", 1, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L21 Head 2", "target": TargetSpec(21, "head", 2, keep_ratio=0.0), "ctx": "train_factual_cap"},
        {"desc": "L16 Head 0", "target": TargetSpec(16, "head", 0, keep_ratio=0.0), "ctx": "ood_distractor_reasoning"},
        {"desc": "L22 Head 2", "target": TargetSpec(22, "head", 2, keep_ratio=0.0), "ctx": "train_factual_cap"},
    ]

    # مجموعه ارزیابی گسترش یافته N=36
    expanded_held_out_36 = [
        {"desc": "L0 MLP", "target": TargetSpec(0, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L0 H1", "target": TargetSpec(0, "head", 1), "ctx": "train_factual_cap"},
        {"desc": "L2 MLP", "target": TargetSpec(2, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L2 H1", "target": TargetSpec(2, "head", 1), "ctx": "ood_syntactic_agreement"},
        {"desc": "L3 MLP", "target": TargetSpec(3, "mlp", 0), "ctx": "train_factual_cap"},
        {"desc": "L3 H0", "target": TargetSpec(3, "head", 0), "ctx": "ood_distractor_reasoning"},
        {"desc": "L4 MLP", "target": TargetSpec(4, "mlp", 0), "ctx": "train_factual_cap"},
        {"desc": "L4 H0", "target": TargetSpec(4, "head", 0), "ctx": "ood_distractor_reasoning"},
        {"desc": "L6 MLP", "target": TargetSpec(6, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L6 H2", "target": TargetSpec(6, "head", 2), "ctx": "train_factual_cap"},
        {"desc": "L7 MLP", "target": TargetSpec(7, "mlp", 0), "ctx": "ood_distractor_reasoning"},
        {"desc": "L7 H1", "target": TargetSpec(7, "head", 1), "ctx": "train_factual_cap"},
        {"desc": "L9 MLP", "target": TargetSpec(9, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L9 H3", "target": TargetSpec(9, "head", 3), "ctx": "ood_distractor_reasoning"},
        {"desc": "L10 MLP", "target": TargetSpec(10, "mlp", 0), "ctx": "train_factual_cap"},
        {"desc": "L10 H0", "target": TargetSpec(10, "head", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L11 MLP", "target": TargetSpec(11, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L11 H2", "target": TargetSpec(11, "head", 2), "ctx": "train_factual_cap"},
        {"desc": "L12 MLP", "target": TargetSpec(12, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L12 H3", "target": TargetSpec(12, "head", 3), "ctx": "ood_syntactic_agreement"},
        {"desc": "L13 MLP", "target": TargetSpec(13, "mlp", 0), "ctx": "ood_distractor_reasoning"},
        {"desc": "L13 H1", "target": TargetSpec(13, "head", 1), "ctx": "train_factual_cap"},
        {"desc": "L14 MLP", "target": TargetSpec(14, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L14 H2", "target": TargetSpec(14, "head", 2), "ctx": "ood_distractor_reasoning"},
        {"desc": "L15 MLP", "target": TargetSpec(15, "mlp", 0), "ctx": "train_factual_cap"},
        {"desc": "L15 H0", "target": TargetSpec(15, "head", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L17 MLP", "target": TargetSpec(17, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L17 H3", "target": TargetSpec(17, "head", 3), "ctx": "train_factual_cap"},
        {"desc": "L20 MLP", "target": TargetSpec(20, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L20 H1", "target": TargetSpec(20, "head", 1), "ctx": "ood_distractor_reasoning"},
        {"desc": "L21 H0", "target": TargetSpec(21, "head", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L22 MLP", "target": TargetSpec(22, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L23 MLP", "target": TargetSpec(23, "mlp", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L10 H2", "target": TargetSpec(10, "head", 2), "ctx": "ood_distractor_reasoning"},
        {"desc": "L6 H0", "target": TargetSpec(6, "head", 0), "ctx": "ood_syntactic_agreement"},
        {"desc": "L15 H3", "target": TargetSpec(15, "head", 3), "ctx": "ood_distractor_reasoning"}
    ]

    print("\n" + "=" * 195)
    print("🔬 MILESTONE 17.18: PARETO SWEEP & STATISTICAL ROBUSTNESS AUDIT (N=36)")
    print("   Protocol: N=36 Expanded Targets | Full Beta Sweep [0, 2.0] | Paired Bootstrap 95% CI")
    print("=" * 195)

    print("\n📦 Training Scaled Empirical Calibrator on 18 Disjoint Samples...")
    calibrator = ScaledEmpiricalCalibrator()
    for item in calibration_pool_18:
        target = item["target"]
        ctx = item["ctx"]
        items = corpus[ctx]
        scm = PureEpistemicSelfGraph(prior_variance=0.5, base_aleatoric=0.005)
        for d in [0.15, 0.35, 0.60]:
            probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
            obs = sensors.measure_grounded_outcomes(items, targets=[probe], noise_sigma=0.005)
            scm.update(target.key, ctx, d, obs)
        pred_mu, sig_pred, sig_epi, _ = scm.predict([target], ctx)
        true_delta = sensors.measure_grounded_outcomes(items, targets=[target], noise_sigma=0.0)
        calibrator.store_sample(pred_mu, sig_pred, sig_epi, true_delta)

    calibrator.train_calibrator()
    print(f"📦 Calibration Status: {'✅ SUCCESS (18 Samples Fitted)' if calibrator.is_fitted else '⚠️ FALLBACK'}")

    def resolve_governor(mu: float, sigma_pred: float, beliefs: Dict[str, float]) -> str:
        if beliefs["P_hazard"] >= 0.50: return "VETO_HAZARD"
        if beliefs["P_gain"] >= 0.50: return "APPROVED"
        if beliefs["P_zero"] >= 0.50: return "VETO_ZERO_GAIN"
        if sigma_pred > 0.65 or max(beliefs.values()) < 0.40: return "VETO_EPISTEMIC"
        best_class = max(beliefs, key=beliefs.get)
        if best_class == "P_gain": return "APPROVED"
        if best_class == "P_hazard": return "VETO_HAZARD"
        return "VETO_ZERO_GAIN"

    policies = ["DECISION_EVOI", "PURE_INFO", "MATCHED_RANDOM", "NO_PROBE"]
    per_target_results = {p: {"regret": [], "risk": [], "sigma": [], "sfr": [], "olr": []} for p in policies}
    op_noise = 0.005

    print("\n📦 Evaluating Policies on N=36 Disjoint Targets...")
    for idx, item in enumerate(expanded_held_out_36):
        target = item["target"]
        ctx = item["ctx"]
        items = corpus[ctx]
        gt_delta = sensors.measure_grounded_outcomes(items, targets=[target], noise_sigma=0.0)
        oracle_action, oracle_u = DecoupledUtility.get_omniscient_oracle(gt_delta)

        for pol in policies:
            scm = PureEpistemicSelfGraph(prior_variance=0.5, base_aleatoric=0.005)
            incurred_risk = 0.0
            executed_doses = []

            if pol == "NO_PROBE":
                pass
            elif pol == "MATCHED_RANDOM":
                np.random.seed(SEED + hash(target.key) % 1000)
                fair_random_doses = sorted(np.random.uniform(0.10, 0.75, size=3).tolist())
                for rd in fair_random_doses:
                    probe_r = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - rd)
                    obs_r = sensors.measure_grounded_outcomes(items, targets=[probe_r], noise_sigma=op_noise)
                    incurred_risk += max(0.0, -obs_r)
                    scm.update(target.key, ctx, rd, obs_r)
                    executed_doses.append(rd)
            else:
                engine = IsolatedPolicyEngine(scm, calibrator, policy_mode=pol)
                for _ in range(4):
                    d, should_stop, ep_state, reason, beliefs = engine.evaluate_step(target, ctx, executed_doses)
                    if should_stop: break
                    probe = TargetSpec(target.layer, target.component, target.index, keep_ratio=1.0 - d)
                    obs = sensors.measure_grounded_outcomes(items, targets=[probe], noise_sigma=op_noise)
                    incurred_risk += max(0.0, -obs)
                    scm.update(target.key, ctx, d, obs)
                    executed_doses.append(d)

            pred_mu, sig_pred, sig_epi, _ = scm.predict([target], ctx)
            beliefs = calibrator.predict_beliefs(pred_mu, sig_pred, sig_epi)
            verdict = resolve_governor(pred_mu, sig_pred, beliefs)
            u = DecoupledUtility.get_utility(verdict, gt_delta)
            regret = oracle_u - u

            per_target_results[pol]["regret"].append(regret)
            per_target_results[pol]["risk"].append(incurred_risk)
            per_target_results[pol]["sigma"].append(sig_pred)

            is_sfr = 1 if (oracle_action == "VETO_HAZARD" and verdict == "APPROVED") else 0
            is_olr = 1 if (oracle_action == "APPROVED" and "VETO" in verdict) else 0
            per_target_results[pol]["sfr"].append(is_sfr)
            per_target_results[pol]["olr"].append(is_olr)

        if (idx + 1) % 6 == 0:
            print(f"   [{idx + 1:02d}/36] Evaluated...")

    # ==============================================================================
    # 7. SUMMARY REPORT & METRICS
    # ==============================================================================
    print("\n" + "=" * 165)
    print("📊 1. AGGREGATE PERFORMANCE AUDIT (N=36 TARGETS)")
    print("-" * 165)
    print(f"{'Policy Architecture':<18} | {'Mean Regret':<12} | {'Mean Risk':<11} | {'Mean σ':<9} | {'Loss (β=0.1)':<13} | {'Loss (β=0.5)':<13} | {'Loss (β=1.0)':<13} | {'OLR (%)':<9} | {'SFR (%)'}")
    print("-" * 165)

    n_samples = len(expanded_held_out_36)
    for pol in policies:
        m_reg = float(np.mean(per_target_results[pol]["regret"]))
        m_risk = float(np.mean(per_target_results[pol]["risk"]))
        m_sig = float(np.mean(per_target_results[pol]["sigma"]))
        l_01 = m_reg + 0.1 * m_risk
        l_05 = m_reg + 0.5 * m_risk
        l_10 = m_reg + 1.0 * m_risk
        olr = float(np.mean(per_target_results[pol]["olr"])) * 100.0
        sfr = float(np.mean(per_target_results[pol]["sfr"])) * 100.0
        print(f"{pol:<18} | {m_reg:<12.4f} | {m_risk:<11.4f} | {m_sig:<9.4f} | {l_01:<13.4f} | {l_05:<13.4f} | {l_10:<13.4f} | {olr:<9.1f} | {sfr:<7.1f}")
    print("=" * 165)

    # ==============================================================================
    # 8. PARETO FRONTIER SWEEP OVER BETA
    # ==============================================================================
    beta_values = [0.0, 0.05, 0.10, 0.20, 0.30, 0.50, 0.75, 1.00, 1.50, 2.00]
    print("\n" + "=" * 135)
    print("📈 2. RISK-REGRET PARETO SWEEP OVER BETA (Mean Loss: Regret + β * Risk)")
    print("-" * 135)
    header = f"{'β Value':<10} | " + " | ".join([f"{p:<16}" for p in policies]) + " | Best Policy"
    print(header)
    print("-" * 135)

    pareto_sweep_results = {}
    for beta in beta_values:
        row_losses = {}
        for pol in policies:
            loss = np.mean(per_target_results[pol]["regret"]) + beta * np.mean(per_target_results[pol]["risk"])
            row_losses[pol] = float(loss)
        best_p = min(row_losses, key=row_losses.get)
        pareto_sweep_results[str(beta)] = {"losses": row_losses, "best": best_p}
        row_str = f"{beta:<10.2f} | " + " | ".join([f"{row_losses[p]:<16.4f}" for p in policies]) + f" | 🏆 {best_p}"
        print(row_str)
    print("=" * 135)

    # ==============================================================================
    # 9. PAIRED STATISTICAL ANALYSIS & BOOTSTRAP (DECISION_EVOI vs. BASELINES)
    # ==============================================================================
    print("\n" + "=" * 135)
    print("🔬 3. PAIRED STATISTICAL INFERENCE: DECISION_EVOI vs. BASELINES (N=36)")
    print("-" * 135)
    print(f"{'Baseline Comparison':<22} | {'Paired ΔRegret [95% CI]':<26} | {'Paired ΔRisk [95% CI]':<26} | {'Cohen d (Risk)':<14} | {'Risk Win-Rate'}")
    print("-" * 135)

    n_boot = 2000
    evoi_regret = np.array(per_target_results["DECISION_EVOI"]["regret"])
    evoi_risk = np.array(per_target_results["DECISION_EVOI"]["risk"])

    statistical_inference = {}

    for base in ["PURE_INFO", "MATCHED_RANDOM", "NO_PROBE"]:
        base_regret = np.array(per_target_results[base]["regret"])
        base_risk = np.array(per_target_results[base]["risk"])

        d_regret = evoi_regret - base_regret
        d_risk = evoi_risk - base_risk

        # Bootstrap CI 95%
        boot_d_reg = [np.mean(d_regret[np.random.choice(n_samples, n_samples, replace=True)]) for _ in range(n_boot)]
        boot_d_risk = [np.mean(d_risk[np.random.choice(n_samples, n_samples, replace=True)]) for _ in range(n_boot)]

        ci_reg = (float(np.percentile(boot_d_reg, 2.5)), float(np.percentile(boot_d_reg, 97.5)))
        ci_risk = (float(np.percentile(boot_d_risk, 2.5)), float(np.percentile(boot_d_risk, 97.5)))

        # Cohen's d for Risk reduction
        std_pool_risk = np.sqrt((np.var(evoi_risk, ddof=1) + np.var(base_risk, ddof=1)) / 2.0)
        cohen_d_risk = float(np.mean(d_risk) / max(std_pool_risk, 1e-6))

        # Win Rate (% targets where EVOI had less or equal risk)
        win_rate_risk = float(np.mean(evoi_risk <= base_risk)) * 100.0

        statistical_inference[f"EVOI_vs_{base}"] = {
            "mean_delta_regret": float(np.mean(d_regret)), "ci_regret_95": ci_reg,
            "mean_delta_risk": float(np.mean(d_risk)), "ci_risk_95": ci_risk,
            "cohen_d_risk": cohen_d_risk, "win_rate_risk": win_rate_risk
        }

        reg_ci_str = f"{np.mean(d_regret):+.4f} [{ci_reg[0]:.4f}, {ci_reg[1]:.4f}]"
        risk_ci_str = f"{np.mean(d_risk):+.4f} [{ci_risk[0]:.4f}, {ci_risk[1]:.4f}]"
        print(f"vs. {base:<18} | {reg_ci_str:<26} | {risk_ci_str:<26} | {cohen_d_risk:<14.3f} | {win_rate_risk:.1f}%")

    print("=" * 135 + "\n")

    telemetry = {
        "per_target_results": per_target_results,
        "pareto_sweep": pareto_sweep_results,
        "statistical_inference": statistical_inference
    }
    with open("m17_18_pareto_results.json", "w", encoding="utf-8") as f:
        json.dump(to_jsonable(telemetry), f, indent=2)
    print("📁 Comprehensive telemetry saved to 'm17_18_pareto_results.json'.")


if __name__ == "__main__":
    run_milestone_17_18()

🔄 Initializing Sensors (Qwen/Qwen2.5-0.5B) on CUDA...


config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer

🔬 MILESTONE 17.18: PARETO SWEEP & STATISTICAL ROBUSTNESS AUDIT (N=36)
   Protocol: N=36 Expanded Targets | Full Beta Sweep [0, 2.0] | Paired Bootstrap 95% CI

📦 Training Scaled Empirical Calibrator on 18 Disjoint Samples...
📦 Calibration Status: ✅ SUCCESS (18 Samples Fitted)

📦 Evaluating Policies on N=36 Disjoint Targets...
   [06/36] Evaluated...
   [12/36] Evaluated...
   [18/36] Evaluated...
   [24/36] Evaluated...
   [30/36] Evaluated...
   [36/36] Evaluated...

📊 1. AGGREGATE PERFORMANCE AUDIT (N=36 TARGETS)
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
Policy Architecture | Mean Regret  | Mean Risk   | Mean σ    | Loss (β=0.1)  | Loss (β=0.5)  | Loss (β=1.0)  | OLR (%)   | SFR (%)
---------------------------------------------------------------------------------------------------------------------